# Day 04 - Column Operations

**Topic:** Column Operations  
**Main environment:** Microsoft Fabric Notebook + Fabric Lakehouse  
**Focus:** ใช้ PySpark DataFrame column operations เพื่อ transform raw data ให้เป็น standardized DataFrame

วันนี้เราจะฝึก operation ที่ใช้บ่อยมากใน PySpark เช่น `select`, `filter`, `where`, `withColumn`, `drop`, `alias`, `cast`, และ `when` / `otherwise`

แนวคิดหลักของวันนี้คือ:

> Raw data มักเข้ามาในรูปแบบที่ยังไม่พร้อมใช้ เราต้องเลือก column, เปลี่ยนชื่อ, cast type, normalize ค่า, สร้าง derived columns และ flag record ที่ควรตรวจต่อ


## Goal of the day

1. ใช้ `select`, `alias`, `withColumn`, `drop` เพื่อจัดรูป DataFrame ได้
2. ใช้ `filter` / `where` เพื่อเลือก records ตามเงื่อนไขได้
3. ใช้ `cast` เพื่อแปลง data type อย่างระวังได้
4. ใช้ `when` / `otherwise` เพื่อสร้าง flag หรือ business rule column ได้
5. Transform raw transaction data ให้เป็น standardized DataFrame ที่พร้อมต่อยอดไป cleansing / Silver layer ได้


## Why it matters in production

ใน production pipeline, Column Operations คือ building blocks หลักของงาน Data Engineering เพราะ:

- Raw data มักมี column name ไม่สม่ำเสมอ เช่น `TxnID`, `transaction_id`, `Transaction ID`
- Source data มักส่งตัวเลขมาเป็น string เช่น `"1,500.50"` หรือ `"not_available"`
- Business logic ต้องสร้าง column ใหม่ เช่น `amount_band`, `is_valid_amount`, `record_quality_status`
- การเลือกเฉพาะ column ที่จำเป็นช่วยลด data movement และทำให้ pipeline อ่านง่ายขึ้น
- การ cast / normalize / flag records เป็นพื้นฐานของ Silver transformation และ Data Quality

Production takeaway:

> Column operations ไม่ใช่แค่ syntax แต่เป็นจุดที่เราเปลี่ยน raw data ให้เริ่มมี standard, type, meaning และ quality signal


## Key concepts

| Concept | Meaning |
|---|---|
| `select` | เลือก column ที่ต้องการ และสร้าง expression ใหม่ได้ |
| `F.col()` | อ้างถึง column แบบ explicit เหมาะกับ expression |
| `alias` | ตั้งชื่อ column ใหม่ใน `select` หรือ aggregation |
| `filter` / `where` | กรอง rows ตาม condition ใช้งานใกล้เคียงกัน |
| `withColumn` | เพิ่ม column ใหม่ หรือ overwrite column เดิมด้วย expression ใหม่ |
| `drop` | ลบ column ที่ไม่ต้องการออกจาก DataFrame |
| `cast` | แปลง data type เช่น string → double, string → date |
| `when` / `otherwise` | สร้าง conditional logic คล้าย `CASE WHEN` ใน SQL |
| `F.lit()` | สร้างค่าคงที่ให้ทุก row ใน column นั้น ๆ |
| Column expression | Logic ที่ใช้ columns, functions, operators และ conditions เพื่อคำนวณค่า |


## Concept explanation

### Column Operations คืออะไร?

ใน PySpark เราไม่ได้แก้ค่าใน DataFrame เดิมแบบ mutable object แต่เราสร้าง DataFrame ใหม่จาก transformation เดิมเสมอ

ตัวอย่างเช่น:

```python
df_clean = df_raw.withColumn("amount", F.col("amount_raw").cast("double"))
```

บรรทัดนี้ไม่ได้แก้ `df_raw` ตรง ๆ แต่สร้าง DataFrame ใหม่ชื่อ `df_clean`

### Column object vs column name string

ใน PySpark มี pattern ที่เจอบ่อย:

```python
F.col("amount")
```

`F.col("amount")` คือ Column object ที่ใช้ทำ expression ได้ เช่น:

```python
F.col("amount") * 1.07
F.col("status") == "success"
F.col("amount").cast("double")
```

ส่วน string เช่น `"amount"` มักใช้ตอน select column แบบง่าย ๆ:

```python
df.select("transaction_id", "amount")
```

### `select` vs `withColumn`

ใช้ `select` เมื่ออยาก:

- เลือกเฉพาะ columns ที่ต้องการ
- rename หลาย columns พร้อมกัน
- สร้าง output schema ให้ชัดเจน

ใช้ `withColumn` เมื่ออยาก:

- เพิ่ม column ใหม่
- แปลง column เดิม
- สร้าง flag หรือ derived column เพิ่มทีละ step

### `when` / `otherwise`

`when` / `otherwise` ใช้สร้าง conditional column เช่น:

```python
F.when(F.col("amount") > 1000, "high").otherwise("normal")
```

เทียบกับ SQL ได้ใกล้เคียงกับ:

```sql
CASE WHEN amount > 1000 THEN 'high' ELSE 'normal' END
```

### Mindset สำคัญ

> Column operation ส่วนใหญ่ยังเป็น Transformation เหมือน Day 02 คือ Spark จะ build plan ก่อน และ execute จริงตอนเจอ action เช่น `.show()`, `.count()`, `.write`


## Mermaid diagram: Raw columns to standardized columns

```mermaid
flowchart LR
    A[Raw DataFrame] --> B[Select and Alias Columns]
    B --> C[Normalize String Values]
    C --> D[Cast Data Types]
    D --> E[Create Derived Columns]
    E --> F[Create Quality Flags]
    F --> G[Drop Temporary Columns]
    G --> H[Standardized DataFrame]
```

Key idea:

> Column operations คือ flow การค่อย ๆ เปลี่ยน raw columns ให้เป็น columns ที่พร้อมใช้งานและตรวจสอบได้


## Setup / imports


In [1]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

StatementMeta(, 099411cd-38ba-4ef5-811c-a439b6e1ec06, 3, Finished, Available, Finished, False)

## Create mock data

Dataset นี้ตั้งใจให้มี raw values ที่เจอได้จริง เช่น:

- amount เป็น string
- amount มี comma
- amount บางรายการเป็น `not_available`
- status / payment method มีตัวพิมพ์ใหญ่เล็กและ whitespace ไม่สม่ำเสมอ
- customer id มี leading zero ซึ่งไม่ควร cast เป็น integer ทันทีถ้าเป็น business key


In [2]:
transactions_raw_data = [
    ("T001", "001", "2026-05-01", "1,500.50", " Credit_Card ", "SUCCESS", "web"),
    ("T002", "002", "2026-05-02", "0.00", "PromptPay", "failed", "mobile"),
    ("T003", "003", "invalid_date", "450.00", "bank_transfer", "Success", "branch"),
    ("T004", "999", "2026-05-03", "not_available", "credit_card", "pending", "web"),
    ("T005", "006", "2026-05-04", "780.25", " promptpay ", "success", "mobile"),
    ("T006", "007", "2026-05-05", "2500", "cash", "SUCCESS", "branch"),
    ("T007", "008", "2026-05-06", "-100.00", "credit_card", "success", "web"),
]

transactions_raw_schema = T.StructType([
    T.StructField("transaction_id_raw", T.StringType(), True),
    T.StructField("customer_id_raw", T.StringType(), True),
    T.StructField("transaction_date_raw", T.StringType(), True),
    T.StructField("amount_raw", T.StringType(), True),
    T.StructField("payment_method_raw", T.StringType(), True),
    T.StructField("status_raw", T.StringType(), True),
    T.StructField("source_system", T.StringType(), True),
])

df_transactions_raw = spark.createDataFrame(transactions_raw_data, transactions_raw_schema)

df_transactions_raw.show(truncate=False)
df_transactions_raw.printSchema()

StatementMeta(, 099411cd-38ba-4ef5-811c-a439b6e1ec06, 4, Finished, Available, Finished, False)

+------------------+---------------+--------------------+-------------+------------------+----------+-------------+
|transaction_id_raw|customer_id_raw|transaction_date_raw|amount_raw   |payment_method_raw|status_raw|source_system|
+------------------+---------------+--------------------+-------------+------------------+----------+-------------+
|T001              |001            |2026-05-01          |1,500.50     | Credit_Card      |SUCCESS   |web          |
|T002              |002            |2026-05-02          |0.00         |PromptPay         |failed    |mobile       |
|T003              |003            |invalid_date        |450.00       |bank_transfer     |Success   |branch       |
|T004              |999            |2026-05-03          |not_available|credit_card       |pending   |web          |
|T005              |006            |2026-05-04          |780.25       | promptpay        |success   |mobile       |
|T006              |007            |2026-05-05          |2500         |c

## PySpark code examples

ใน section นี้จะไล่จาก basic column selection ไปจนถึง standardized DataFrame:

1. `select` / `alias`
2. `filter` / `where`
3. `withColumn`
4. safe `cast`
5. `when` / `otherwise`
6. `drop`
7. write result to Fabric Lakehouse table


### Example 1: Select columns and rename with alias

ใช้ `select` เพื่อกำหนด output columns ที่ต้องการ และใช้ `.alias()` เพื่อ rename column ให้อ่านง่ายขึ้น

Pattern นี้ดีมากเวลาเปลี่ยนจาก raw naming ให้เป็น standardized naming


In [3]:
df_selected = df_transactions_raw.select(
    F.col("transaction_id_raw").alias("transaction_id"),
    F.col("customer_id_raw").alias("customer_id"),
    F.col("transaction_date_raw").alias("transaction_date_text"),
    F.col("amount_raw").alias("amount_text_raw"),
    F.col("payment_method_raw").alias("payment_method_text"),
    F.col("status_raw").alias("status_text"),
    "source_system"
)

df_selected.show(truncate=False)

StatementMeta(, 099411cd-38ba-4ef5-811c-a439b6e1ec06, 5, Finished, Available, Finished, False)

+--------------+-----------+---------------------+---------------+-------------------+-----------+-------------+
|transaction_id|customer_id|transaction_date_text|amount_text_raw|payment_method_text|status_text|source_system|
+--------------+-----------+---------------------+---------------+-------------------+-----------+-------------+
|T001          |001        |2026-05-01           |1,500.50       | Credit_Card       |SUCCESS    |web          |
|T002          |002        |2026-05-02           |0.00           |PromptPay          |failed     |mobile       |
|T003          |003        |invalid_date         |450.00         |bank_transfer      |Success    |branch       |
|T004          |999        |2026-05-03           |not_available  |credit_card        |pending    |web          |
|T005          |006        |2026-05-04           |780.25         | promptpay         |success    |mobile       |
|T006          |007        |2026-05-05           |2500           |cash               |SUCCESS   

### Example 2: Filter rows with `filter` and `where`

`filter` และ `where` ใช้กรอง rows ตาม condition ได้ใกล้เคียงกัน

ใน practice สามารถเลือกใช้ได้ตาม readability ของทีม แต่ควรใช้ให้ consistent ใน notebook หรือ project เดียวกัน


In [4]:
df_web_transactions = df_selected.filter(F.col("source_system") == "web")
df_mobile_transactions = df_selected.where(F.col("source_system") == "mobile")

print("Web transactions")
df_web_transactions.show(truncate=False)

print("Mobile transactions")
df_mobile_transactions.show(truncate=False)

StatementMeta(, 099411cd-38ba-4ef5-811c-a439b6e1ec06, 6, Finished, Available, Finished, False)

Web transactions
+--------------+-----------+---------------------+---------------+-------------------+-----------+-------------+
|transaction_id|customer_id|transaction_date_text|amount_text_raw|payment_method_text|status_text|source_system|
+--------------+-----------+---------------------+---------------+-------------------+-----------+-------------+
|T001          |001        |2026-05-01           |1,500.50       | Credit_Card       |SUCCESS    |web          |
|T004          |999        |2026-05-03           |not_available  |credit_card        |pending    |web          |
|T007          |008        |2026-05-06           |-100.00        |credit_card        |success    |web          |
+--------------+-----------+---------------------+---------------+-------------------+-----------+-------------+

Mobile transactions
+--------------+-----------+---------------------+---------------+-------------------+-----------+-------------+
|transaction_id|customer_id|transaction_date_text|amount_t

### Example 3: Normalize string columns with `withColumn`

ใช้ `withColumn` เพื่อสร้าง column ใหม่จาก column เดิม

ตัวอย่างนี้ normalize ค่า string โดย:

- `trim` ตัด whitespace หน้า/หลัง
- `lower` ทำให้เป็น lower case
- `regexp_replace` เปลี่ยน comma ออกจาก amount text ก่อน cast

> วันนี้ไม่ได้ลงลึก string functions มาก เพราะ Day 08 จะมี topic Date / Timestamp / String Functions แยก แต่ใช้เท่าที่จำเป็นเพื่อให้เห็น column expression


In [5]:
df_normalized_text = (
    df_selected
    .withColumn("payment_method", F.lower(F.trim(F.col("payment_method_text"))))
    .withColumn("status", F.lower(F.trim(F.col("status_text"))))
    .withColumn("amount_text", F.regexp_replace(F.trim(F.col("amount_text_raw")), ",", ""))
)

df_normalized_text.select(
    "transaction_id",
    "amount_text_raw",
    "amount_text",
    "payment_method_text",
    "payment_method",
    "status_text",
    "status"
).show(truncate=False)

StatementMeta(, 099411cd-38ba-4ef5-811c-a439b6e1ec06, 7, Finished, Available, Finished, False)

+--------------+---------------+-------------+-------------------+--------------+-----------+-------+
|transaction_id|amount_text_raw|amount_text  |payment_method_text|payment_method|status_text|status |
+--------------+---------------+-------------+-------------------+--------------+-----------+-------+
|T001          |1,500.50       |1500.50      | Credit_Card       |credit_card   |SUCCESS    |success|
|T002          |0.00           |0.00         |PromptPay          |promptpay     |failed     |failed |
|T003          |450.00         |450.00       |bank_transfer      |bank_transfer |Success    |success|
|T004          |not_available  |not_available|credit_card        |credit_card   |pending    |pending|
|T005          |780.25         |780.25       | promptpay         |promptpay     |success    |success|
|T006          |2500           |2500         |cash               |cash          |SUCCESS    |success|
|T007          |-100.00        |-100.00      |credit_card        |credit_card   |s

### Example 4: Safe cast from string to numeric/date

การ cast raw string โดยตรงอาจเสี่ยง เพราะ source อาจมีค่าอย่าง `not_available` หรือ format แปลก ๆ

ตัวอย่างนี้ใช้ pattern ที่ปลอดภัยขึ้น:

1. ตรวจว่า amount เป็น numeric string ด้วย `rlike`
2. ถ้าใช่ค่อย cast เป็น `double`
3. ถ้าไม่ใช่ให้เป็น `null`

สำหรับ `transaction_date` เราตรวจ format เบื้องต้นก่อนแปลงเป็น date


In [6]:
df_typed = (
    df_normalized_text
    .withColumn(
        "amount",
        F.when(
            F.col("amount_text").rlike(r"^-?\d+(\.\d+)?$"),
            F.col("amount_text").cast(T.DoubleType())
        ).otherwise(F.lit(None).cast(T.DoubleType()))
    )
    .withColumn(
        "transaction_date",
        F.when(
            F.col("transaction_date_text").rlike(r"^\d{4}-\d{2}-\d{2}$"),
            F.to_date(F.col("transaction_date_text"), "yyyy-MM-dd")
        ).otherwise(F.lit(None).cast(T.DateType()))
    )
)

# Tip:
# - .rlike(r"^-?\d+(\.\d+)?$") checks if the value is numeric, such as 100, -100, or 100.50.
# - .rlike(r"^\d{4}-\d{2}-\d{2}$") checks if the value follows yyyy-MM-dd date format.
# - None is inferred as NullType, so cast it explicitly to avoid an ambiguous column type.

df_typed.select(
    "transaction_id",
    "amount_text_raw",
    "amount_text",
    "amount",
    "transaction_date_text",
    "transaction_date"
).show(truncate=False)

df_typed.printSchema()

StatementMeta(, 099411cd-38ba-4ef5-811c-a439b6e1ec06, 8, Finished, Available, Finished, False)

+--------------+---------------+-------------+------+---------------------+----------------+
|transaction_id|amount_text_raw|amount_text  |amount|transaction_date_text|transaction_date|
+--------------+---------------+-------------+------+---------------------+----------------+
|T001          |1,500.50       |1500.50      |1500.5|2026-05-01           |2026-05-01      |
|T002          |0.00           |0.00         |0.0   |2026-05-02           |2026-05-02      |
|T003          |450.00         |450.00       |450.0 |invalid_date         |NULL            |
|T004          |not_available  |not_available|NULL  |2026-05-03           |2026-05-03      |
|T005          |780.25         |780.25       |780.25|2026-05-04           |2026-05-04      |
|T006          |2500           |2500         |2500.0|2026-05-05           |2026-05-05      |
|T007          |-100.00        |-100.00      |-100.0|2026-05-06           |2026-05-06      |
+--------------+---------------+-------------+------+-----------------

### Example 5: Create derived columns with `when` / `otherwise`

ตัวอย่างนี้สร้าง column ใหม่เพื่อใช้เป็น quality signal และ business classification:

- `is_valid_amount`
- `amount_band`
- `is_success_transaction`
- `record_quality_status`

Pattern นี้เจอบ่อยมากใน cleansing, validation, and standardization step


In [7]:
df_with_flags = (
    df_typed
    .withColumn(
        "is_valid_amount",
        F.when(F.col("amount").isNotNull() & (F.col("amount") > 0), F.lit(True))
         .otherwise(F.lit(False))
    )
    .withColumn(
        "amount_band",
        F.when(F.col("amount").isNull(), F.lit("unknown"))
         .when(F.col("amount") >= 1000, F.lit("high"))
         .when(F.col("amount") > 0, F.lit("normal"))
         .otherwise(F.lit("invalid"))
    )
    .withColumn(
        "is_success_transaction",
        F.when(F.col("status") == "success", F.lit(True)).otherwise(F.lit(False))
    )
    .withColumn(
        "record_quality_status",
        F.when(F.col("transaction_id").isNull(), F.lit("reject_missing_key"))
         .when(F.col("transaction_date").isNull(), F.lit("review_invalid_date"))
         .when(~F.col("is_valid_amount"), F.lit("review_invalid_amount"))
         .otherwise(F.lit("pass"))
    )
)

df_with_flags.select(
    "transaction_id",
    "amount",
    "is_valid_amount",
    "amount_band",
    "status",
    "is_success_transaction",
    "transaction_date",
    "record_quality_status"
).show(truncate=False)

StatementMeta(, 099411cd-38ba-4ef5-811c-a439b6e1ec06, 9, Finished, Available, Finished, False)

+--------------+------+---------------+-----------+-------+----------------------+----------------+---------------------+
|transaction_id|amount|is_valid_amount|amount_band|status |is_success_transaction|transaction_date|record_quality_status|
+--------------+------+---------------+-----------+-------+----------------------+----------------+---------------------+
|T001          |1500.5|true           |high       |success|true                  |2026-05-01      |pass                 |
|T002          |0.0   |false          |invalid    |failed |false                 |2026-05-02      |review_invalid_amount|
|T003          |450.0 |true           |normal     |success|true                  |NULL            |review_invalid_date  |
|T004          |NULL  |false          |unknown    |pending|false                 |2026-05-03      |review_invalid_amount|
|T005          |780.25|true           |normal     |success|true                  |2026-05-04      |pass                 |
|T006          |2500.0|t

### Example 6: Drop temporary raw columns

หลังจากสร้าง standardized columns แล้ว เราสามารถ drop temporary columns ที่ไม่อยากให้เป็น output ได้

ข้อควรระวัง: อย่า drop raw columns เร็วเกินไปถ้ายังต้องใช้ debug หรือ explain data quality issue


In [8]:
df_transactions_standardized = (
    df_with_flags
    .withColumn("processing_timestamp", F.current_timestamp())
    .withColumn("processing_date", F.to_date(F.col("processing_timestamp")))
    .drop(
        "transaction_date_text",
        "amount_text_raw",
        "amount_text",
        "payment_method_text",
        "status_text"
    )
)

df_transactions_standardized.show(truncate=False)
df_transactions_standardized.printSchema()

StatementMeta(, 099411cd-38ba-4ef5-811c-a439b6e1ec06, 10, Finished, Available, Finished, False)

+--------------+-----------+-------------+--------------+-------+------+----------------+---------------+-----------+----------------------+---------------------+--------------------------+---------------+
|transaction_id|customer_id|source_system|payment_method|status |amount|transaction_date|is_valid_amount|amount_band|is_success_transaction|record_quality_status|processing_timestamp      |processing_date|
+--------------+-----------+-------------+--------------+-------+------+----------------+---------------+-----------+----------------------+---------------------+--------------------------+---------------+
|T001          |001        |web          |credit_card   |success|1500.5|2026-05-01      |true           |high       |true                  |pass                 |2026-05-30 13:25:07.042884|2026-05-30     |
|T002          |002        |mobile       |promptpay     |failed |0.0   |2026-05-02      |false          |invalid    |false                 |review_invalid_amount|2026-05-30 13:

### Example 7: Build the same result as one transformation chain

ใน production code เรามักจัด transformation ให้อ่านเป็น flow เดียวชัด ๆ

จุดสำคัญคือยังเป็น lazy transformation chain เหมือน Day 02 จนกว่าจะเจอ action เช่น `.show()`, `.count()`, `.write`


In [9]:
df_final_transactions = (
    df_transactions_raw
    .select(
        F.col("transaction_id_raw").alias("transaction_id"),
        F.col("customer_id_raw").alias("customer_id"),
        F.col("transaction_date_raw").alias("transaction_date_text"),
        F.col("amount_raw").alias("amount_text_raw"),
        F.col("payment_method_raw").alias("payment_method_text"),
        F.col("status_raw").alias("status_text"),
        "source_system"
    )
    .withColumn("payment_method", F.lower(F.trim(F.col("payment_method_text"))))
    .withColumn("status", F.lower(F.trim(F.col("status_text"))))
    .withColumn("amount_text", F.regexp_replace(F.trim(F.col("amount_text_raw")), ",", ""))
    .withColumn(
        "amount",
        F.when(
            F.col("amount_text").rlike(r"^-?\d+(\.\d+)?$"),
            F.col("amount_text").cast(T.DoubleType())
        ).otherwise(F.lit(None).cast(T.DoubleType()))
    )
    .withColumn(
        "transaction_date",
        F.when(
            F.col("transaction_date_text").rlike(r"^\d{4}-\d{2}-\d{2}$"),
            F.to_date(F.col("transaction_date_text"), "yyyy-MM-dd")
        ).otherwise(F.lit(None).cast(T.DateType()))
    )
    .withColumn("is_valid_amount", F.col("amount").isNotNull() & (F.col("amount") > 0))  # Direct boolean expression
    .withColumn(
        "amount_band",
        F.when(F.col("amount").isNull(), F.lit("unknown"))
         .when(F.col("amount") >= 1000, F.lit("high"))
         .when(F.col("amount") > 0, F.lit("normal"))
         .otherwise(F.lit("invalid"))
    )
    .withColumn("is_success_transaction", F.col("status") == "success")  # Direct boolean expression
    .withColumn(
        "record_quality_status",
        F.when(F.col("transaction_id").isNull(), F.lit("reject_missing_key"))
         .when(F.col("transaction_date").isNull(), F.lit("review_invalid_date"))
         .when(~F.col("is_valid_amount"), F.lit("review_invalid_amount"))
         .otherwise(F.lit("pass"))
    )
    .withColumn("processing_timestamp", F.current_timestamp())
    .withColumn("processing_date", F.to_date(F.col("processing_timestamp")))
    .drop("transaction_date_text", "amount_text_raw", "amount_text", "payment_method_text", "status_text")
)

# Tip:
# - Direct boolean expression for True/False flag, equivalent to when(..., F.lit(True)).otherwise(F.lit(False)).
# - F.lit() is optional in many cases, but it makes fixed values explicit as literal values, not column names.

print("Final transformation chain has been defined.")
print("No full data result is returned until an action is called.")

StatementMeta(, 099411cd-38ba-4ef5-811c-a439b6e1ec06, 11, Finished, Available, Finished, False)

Final transformation chain has been defined.
No full data result is returned until an action is called.


ดู execution plan ด้วย `.explain()`


In [10]:
df_final_transactions.explain(mode="formatted")

StatementMeta(, 099411cd-38ba-4ef5-811c-a439b6e1ec06, 12, Finished, Available, Finished, False)

== Physical Plan ==
* Project (5)
+- * Project (4)
   +- * Project (3)
      +- * Project (2)
         +- * Scan ExistingRDD (1)


(1) Scan ExistingRDD [codegen id : 1]
Output [7]: [transaction_id_raw#726, customer_id_raw#727, transaction_date_raw#728, amount_raw#729, payment_method_raw#730, status_raw#731, source_system#732]
Arguments: [transaction_id_raw#726, customer_id_raw#727, transaction_date_raw#728, amount_raw#729, payment_method_raw#730, status_raw#731, source_system#732], MapPartitionsRDD[44] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Project [codegen id : 1]
Output [7]: [transaction_id_raw#726 AS transaction_id#1197, customer_id_raw#727 AS customer_id#1198, transaction_date_raw#728 AS transaction_date_text#1199, source_system#732, lower(trim(payment_method_raw#730, None)) AS payment_method#1210, lower(trim(status_raw#731, None)) AS status#1219, regexp_replace(trim(amount_raw#729, None), ,, , 1) AS amount_text#1229]


Trigger action ด้วย `.show()`


In [11]:
df_final_transactions.show(truncate=False)

StatementMeta(, 099411cd-38ba-4ef5-811c-a439b6e1ec06, 13, Finished, Available, Finished, False)

+--------------+-----------+-------------+--------------+-------+------+----------------+---------------+-----------+----------------------+---------------------+--------------------------+---------------+
|transaction_id|customer_id|source_system|payment_method|status |amount|transaction_date|is_valid_amount|amount_band|is_success_transaction|record_quality_status|processing_timestamp      |processing_date|
+--------------+-----------+-------------+--------------+-------+------+----------------+---------------+-----------+----------------------+---------------------+--------------------------+---------------+
|T001          |001        |web          |credit_card   |success|1500.5|2026-05-01      |true           |high       |true                  |pass                 |2026-05-30 13:25:13.199104|2026-05-30     |
|T002          |002        |mobile       |promptpay     |failed |0.0   |2026-05-02      |false          |invalid    |false                 |review_invalid_amount|2026-05-30 13:

### Example 8: Write standardized DataFrame to Fabric Lakehouse table

ใน Fabric Lakehouse สามารถเขียนผลลัพธ์เป็น Delta table ด้วย `saveAsTable`

> หมายเหตุ: ต้อง attach Lakehouse เข้ากับ Fabric Notebook ก่อนรัน cell นี้ ไม่งั้น `saveAsTable()` อาจ fail เพราะ notebook ไม่มี Lakehouse context


In [12]:
table_name = "day04_transactions_standardized"

(
    df_final_transactions
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(table_name)
)

print(f"Table written successfully: {table_name}")

StatementMeta(, 099411cd-38ba-4ef5-811c-a439b6e1ec06, 14, Finished, Available, Finished, False)

Table written successfully: day04_transactions_standardized


### Example 9: Read table back from Lakehouse

หลังจาก write table แล้ว สามารถ read กลับด้วย `spark.read.table()` เพื่อ validate output ได้


In [13]:
df_saved_transactions = spark.read.table("day04_transactions_standardized")

df_saved_transactions.show(truncate=False)
print("Saved table row count:", df_saved_transactions.count())

StatementMeta(, 099411cd-38ba-4ef5-811c-a439b6e1ec06, 15, Finished, Available, Finished, False)

+--------------+-----------+-------------+--------------+-------+------+----------------+---------------+-----------+----------------------+---------------------+--------------------------+---------------+
|transaction_id|customer_id|source_system|payment_method|status |amount|transaction_date|is_valid_amount|amount_band|is_success_transaction|record_quality_status|processing_timestamp      |processing_date|
+--------------+-----------+-------------+--------------+-------+------+----------------+---------------+-----------+----------------------+---------------------+--------------------------+---------------+
|T007          |008        |web          |credit_card   |success|-100.0|2026-05-06      |false          |invalid    |true                  |review_invalid_amount|2026-05-30 13:25:26.007049|2026-05-30     |
|T002          |002        |mobile       |promptpay     |failed |0.0   |2026-05-02      |false          |invalid    |false                 |review_invalid_amount|2026-05-30 13:

## Quick recap

| Question | Answer |
|---|---|
| `select` ใช้ทำอะไร? | เลือก columns และสร้าง output schema / alias ได้ |
| `withColumn` ใช้ทำอะไร? | เพิ่ม column ใหม่ หรือ overwrite column เดิมด้วย expression ใหม่ |
| `filter` กับ `where` ต่างกันไหม? | ใช้กรอง rows ได้ใกล้เคียงกัน เลือกใช้ตาม style ของทีม |
| ทำไมต้องใช้ `F.col()`? | เพื่ออ้าง column เป็น expression ที่นำไปคำนวณหรือทำ condition ต่อได้ |
| `cast` ต้องระวังอะไร? | raw string อาจ cast แล้วได้ null จึงควร validate format ก่อนเมื่อจำเป็น |
| `when` / `otherwise` คล้ายอะไรใน SQL? | คล้าย `CASE WHEN` |
| Column operations execute ทันทีไหม? | ไม่ทันที ส่วนใหญ่เป็น transformation และจะ execute ตอนเจอ action |


## Exercises


### Exercise 1: Standardize customer columns

สร้าง DataFrame ชื่อ `df_customers_standardized` จาก mock customer raw data

Requirements:

- rename columns ให้เป็น standardized names
- normalize `email` ด้วย `lower` และ `trim`
- normalize `status` ด้วย `lower` และ `trim`
- cast `signup_score_raw` เป็น integer อย่างปลอดภัย
- เพิ่ม column `is_active_customer`

Expected result:

- ได้ DataFrame ที่มี column names อ่านง่าย
- `email` เป็น lower case
- `signup_score` เป็น integer หรือ null ถ้า raw value ไม่ถูกต้อง
- `is_active_customer` เป็น boolean


In [14]:
customers_raw_data = [
    ("C001", "Alice", " ALICE@example.com ", "Active", "85"),
    ("C002", "Bob", "bob@example.com", "inactive", "not_available"),
    ("C003", "Charlie", " CHARLIE@example.com", "ACTIVE", "72"),
    ("C004", "Dara", None, "pending", "60"),
]

customers_raw_schema = T.StructType([
    T.StructField("cust_id_raw", T.StringType(), True),
    T.StructField("cust_name_raw", T.StringType(), True),
    T.StructField("email_raw", T.StringType(), True),
    T.StructField("status_raw", T.StringType(), True),
    T.StructField("signup_score_raw", T.StringType(), True),
])

df_customers_raw = spark.createDataFrame(customers_raw_data, customers_raw_schema)

df_customers_standardized = (
    df_customers_raw
    .select(
        F.col("cust_id_raw").alias("customer_id"),
        F.col("cust_name_raw").alias("customer_name"),
        F.col("email_raw").alias("email_text"),
        F.col("status_raw").alias("status_text"),
        F.col("signup_score_raw").alias("signup_score_text")
    )
    .withColumn("email", F.lower(F.trim(F.col("email_text"))))
    .withColumn("status", F.lower(F.trim(F.col("status_text"))))
    .withColumn(
        "signup_score",
        F.when(F.col("signup_score_text").rlike(r"^\d+$"), F.col("signup_score_text").cast(T.IntegerType()))
         .otherwise(F.lit(None).cast(T.IntegerType()))
    )
    .withColumn("is_active_customer", F.col("status") == "active")
    .drop("email_text", "status_text", "signup_score_text")
)

# Tip:
# - .rlike(r"^\d+$") checks if the value contains only digits, such as 100, 001, or 999.

df_customers_standardized.show(truncate=False)
df_customers_standardized.printSchema()

StatementMeta(, 099411cd-38ba-4ef5-811c-a439b6e1ec06, 16, Finished, Available, Finished, False)

+-----------+-------------+-------------------+--------+------------+------------------+
|customer_id|customer_name|email              |status  |signup_score|is_active_customer|
+-----------+-------------+-------------------+--------+------------+------------------+
|C001       |Alice        |alice@example.com  |active  |85          |true              |
|C002       |Bob          |bob@example.com    |inactive|NULL        |false             |
|C003       |Charlie      |charlie@example.com|active  |72          |true              |
|C004       |Dara         |NULL               |pending |60          |false             |
+-----------+-------------+-------------------+--------+------------+------------------+

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- status: string (nullable = true)
 |-- signup_score: integer (nullable = true)
 |-- is_active_customer: boolean (nullable = true)



### Exercise 2: Create transaction validation flags

ใช้ `df_final_transactions` จาก example ก่อนหน้า แล้วสร้าง DataFrame ชื่อ `df_transaction_validation`

Requirements:

- เพิ่ม column `has_valid_payment_method`
- valid payment methods คือ `credit_card`, `promptpay`, `bank_transfer`, `cash`
- เพิ่ม column `is_review_required`
- review required ถ้า:
  - `record_quality_status` ไม่ใช่ `pass`
  - หรือ `status` เป็น `pending`
- select เฉพาะ columns สำคัญเพื่อใช้ review

Expected result:

- เห็น records ที่ต้อง review ได้ชัดเจน
- ใช้ `filter` เพื่อ preview เฉพาะ records ที่ `is_review_required = true`


In [15]:
valid_payment_methods = ["credit_card", "promptpay", "bank_transfer", "cash"]

df_transaction_validation = (
    df_final_transactions
    .withColumn("has_valid_payment_method", F.col("payment_method").isin(valid_payment_methods))
    .withColumn(
        "is_review_required",
        (F.col("record_quality_status") != "pass") | (F.col("status") == "pending")
    )
    .select(
        "transaction_id",
        "customer_id",
        "amount",
        "payment_method",
        "has_valid_payment_method",
        "status",
        "transaction_date",
        "record_quality_status",
        "is_review_required"
    )
)

df_transaction_validation.show(truncate=False)

print("Records that require review")
df_transaction_validation.filter(F.col("is_review_required") == True).show(truncate=False)

StatementMeta(, 099411cd-38ba-4ef5-811c-a439b6e1ec06, 17, Finished, Available, Finished, False)

+--------------+-----------+------+--------------+------------------------+-------+----------------+---------------------+------------------+
|transaction_id|customer_id|amount|payment_method|has_valid_payment_method|status |transaction_date|record_quality_status|is_review_required|
+--------------+-----------+------+--------------+------------------------+-------+----------------+---------------------+------------------+
|T001          |001        |1500.5|credit_card   |true                    |success|2026-05-01      |pass                 |false             |
|T002          |002        |0.0   |promptpay     |true                    |failed |2026-05-02      |review_invalid_amount|true              |
|T003          |003        |450.0 |bank_transfer |true                    |success|NULL            |review_invalid_date  |true              |
|T004          |999        |NULL  |credit_card   |true                    |pending|2026-05-03      |review_invalid_amount|true              |
|T005 

### Exercise 3: Create a curated success transaction DataFrame

สร้าง DataFrame ชื่อ `df_success_curated` สำหรับ successful transactions ที่ผ่านเงื่อนไขเบื้องต้น

Requirements:

- filter เฉพาะ `status = "success"`
- filter เฉพาะ `is_valid_amount = true`
- select columns:
  - `transaction_id`
  - `customer_id`
  - `transaction_date`
  - `amount`
  - `amount_band`
  - `payment_method`
  - `source_system`
- write เป็น table ชื่อ `day04_success_curated`

Expected result:

- ได้เฉพาะ transaction ที่ status success และ amount valid
- มี table `day04_success_curated` ใน Lakehouse ถ้า attach Lakehouse แล้ว


In [19]:
df_success_curated = (
    df_final_transactions
    .filter(F.col("status") == "success")
    .filter(F.col("is_valid_amount") == True)
    .select(
        "transaction_id",
        "customer_id",
        "transaction_date",
        "amount",
        "amount_band",
        "payment_method",
        "source_system"
    )
)

df_success_curated.show(truncate=False)

(
    df_success_curated
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("day04_success_curated")
)

spark.read.table("day04_success_curated").show(truncate=False)

StatementMeta(, 099411cd-38ba-4ef5-811c-a439b6e1ec06, 21, Finished, Available, Finished, False)

+--------------+-----------+----------------+------+-----------+--------------+-------------+
|transaction_id|customer_id|transaction_date|amount|amount_band|payment_method|source_system|
+--------------+-----------+----------------+------+-----------+--------------+-------------+
|T001          |001        |2026-05-01      |1500.5|high       |credit_card   |web          |
|T003          |003        |NULL            |450.0 |normal     |bank_transfer |branch       |
|T005          |006        |2026-05-04      |780.25|normal     |promptpay     |mobile       |
|T006          |007        |2026-05-05      |2500.0|high       |cash          |branch       |
+--------------+-----------+----------------+------+-----------+--------------+-------------+

+--------------+-----------+----------------+------+-----------+--------------+-------------+
|transaction_id|customer_id|transaction_date|amount|amount_band|payment_method|source_system|
+--------------+-----------+----------------+------+-------

## Common mistakes

1. **คิดว่า `withColumn` แก้ DataFrame เดิม**

   DataFrame ใน Spark เป็น immutable pattern คือ operation จะสร้าง DataFrame ใหม่เสมอ ถ้าไม่ assign กลับ ตัวแปรเดิมจะไม่เปลี่ยน

2. **ใช้ string แทน Column expression ตอนคำนวณ**

   เช่นเขียน `"amount" * 1.07` แทนที่จะใช้ `F.col("amount") * 1.07`
   
   เพราะ ใน PySpark ถ้าจะคำนวณจาก column ต้องใช้ Column expression เช่น `F.col("amount")` เพราะ `"amount"` เป็นแค่ Python string ไม่ใช่ Spark column

3. **Cast raw string ตรง ๆ โดยไม่เช็ค format**

   ถ้า source ส่งค่า `not_available`, empty string, หรือ format แปลก ๆ การ cast อาจได้ null หรือ error ขึ้นกับ runtime/config ควร validate ก่อนในจุดที่สำคัญ

4. **Drop raw columns เร็วเกินไป**

   ถ้า drop raw columns ตั้งแต่ต้น อาจ debug ยากว่า raw value เดิมคืออะไร เช่น cast แล้วกลายเป็น null เพราะค่าเดิม malformed หรือ format ไม่ถูกต้อง

   โดยทั่วไปควรเก็บ raw columns ไว้ระหว่างขั้นตอน cleansing / validation และค่อย drop ตอนสร้าง final curated DataFrame ที่พร้อมใช้งาน

5. **Overwrite column เดิมโดยไม่ตั้งใจ**

   `withColumn("amount", ...)` ถ้ามี column `amount` อยู่แล้ว จะ replace logic เดิมทันที ควรตั้งชื่อ temporary column ให้ชัดเมื่อยังไม่มั่นใจ

6. **Cast business key เป็น integer ทั้งที่ leading zero สำคัญ**

   เช่น `customer_id = "001"` ถ้า cast เป็น int จะกลายเป็น `1` และอาจ join กับระบบอื่นไม่ตรง

7. **เขียน transformation chain ยาวเกินไปจนอ่านยาก**

   ถ้า logic ซับซ้อน แบ่งเป็น step เช่น normalize → type cast → flags → final select จะ review ง่ายกว่า


## Expected Output / Success Criteria

เมื่อจบ Day 04 ควรทำได้:

- ใช้ `select` เพื่อเลือกและจัดลำดับ columns ได้
- ใช้ `.alias()` เพื่อ rename column ได้
- ใช้ `filter` / `where` เพื่อกรอง rows ตาม condition ได้
- ใช้ `withColumn` เพื่อเพิ่มหรือแปลง column ได้
- ใช้ `drop` เพื่อลบ temporary columns ได้
- ใช้ `F.col()` เพื่อสร้าง column expressions ได้ถูกต้อง
- ใช้ `cast` เพื่อแปลง data type เช่น string → double/date ได้
- ใช้ `when` / `otherwise` เพื่อสร้าง conditional flags ได้
- สร้าง standardized transaction DataFrame จาก raw DataFrame ได้
- สร้าง quality flag เช่น `is_valid_amount`, `record_quality_status`, `is_review_required` ได้
- เขียน standardized output เป็น Fabric Lakehouse table ด้วย `saveAsTable()` ได้เมื่อ attach Lakehouse แล้ว
- อธิบายได้ว่า Column Operations ส่วนใหญ่ยังเป็น transformations และ execute จริงเมื่อเจอ action


## Final takeaway

Column Operations คือ daily toolset ของ Data Engineer ใน Spark:

> เลือก column ให้ถูก, ตั้งชื่อให้ชัด, cast type ให้ปลอดภัย, สร้าง derived columns ให้มีความหมาย, และ flag records ที่ต้องตรวจต่อ

สิ่งที่ควรจำจากวันนี้:

- `select` ช่วยจัด output schema ให้ชัด
- `withColumn` ใช้เพิ่มหรือ transform column แต่ต้องระวัง overwrite column เดิม
- `when` / `otherwise` คือ pattern สำคัญสำหรับ business rule และ data quality flag
- อย่า cast business key หรือ raw string แบบไม่คิด เพราะอาจทำให้ข้อมูลเสียความหมาย
- ใน production, column transformation ที่อ่านง่ายและ trace ได้ สำคัญพอ ๆ กับ code ที่ run ผ่าน


## Optional cleanup

รันเฉพาะเมื่อต้องการลบ table ที่สร้างจาก notebook วันนี้


In [20]:
# spark.sql("DROP TABLE IF EXISTS day04_transactions_standardized")
# spark.sql("DROP TABLE IF EXISTS day04_success_curated")

StatementMeta(, 099411cd-38ba-4ef5-811c-a439b6e1ec06, 22, Finished, Available, Finished, False)